In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision import models, transforms
from PIL import Image

# Define maximum images per folder
MAX_IMAGES_PER_FOLDER = 10  # Limit images processed per folder

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define model path and dataset path
checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/HQ2ltst_20241123/best_model_vgg_20241123.pth'
test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ3/train/'

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Define the modified VGG16 model
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        from torchvision.models import VGG16_Weights
        self.vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        # Replace the classifier with a custom binary classification layer
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 2)
        )

    def forward(self, x):
        return self.vgg(x)

# Function to load the trained model
def load_model(checkpoint_path):
    model = VGG16Modified()
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    return model

# Load model
model = load_model(checkpoint_path)
print("Model loaded successfully.")

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# Function to preprocess an image
def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    return image_tensor

# Grad-CAM function
def generate_gradcam(model, image_tensor, target_layer):
    features = []
    grads = []

    # Hook to capture features
    def forward_hook(module, input, output):
        features.append(output)

    # Hook to capture gradients
    def backward_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    # Register hooks
    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_backward_hook(backward_hook)

    # Forward pass
    output = model(image_tensor)
    predicted_class_idx = torch.argmax(output, dim=1).item()

    # Backward pass
    model.zero_grad()
    output[0, predicted_class_idx].backward()

    # Remove hooks
    forward_handle.remove()
    backward_handle.remove()

    # Extract feature maps and gradients
    feature_map = features[0].cpu().detach().numpy()[0]
    gradient = grads[0].cpu().detach().numpy()[0]

    # Compute Grad-CAM
    weights = np.mean(gradient, axis=(1, 2))
    cam = np.zeros(feature_map.shape[1:], dtype=np.float32)

    for i, w in enumerate(weights):
        cam += w * feature_map[i]

    cam = np.maximum(cam, 0)  # Apply ReLU
    cam = cv2.resize(cam, (224, 224))  # Resize to image size
    cam -= cam.min()
    cam /= cam.max() + 1e-8  # Normalize

    return cam, predicted_class_idx

# Function to visualize results
def visualize_results(image_path, gradcam, predicted_class):
    original_image = Image.open(image_path).resize((224, 224), Image.LANCZOS)
    original_image = np.array(original_image)

    # Overlay Grad-CAM on the image
    heatmap = cv2.applyColorMap(np.uint8(255 * gradcam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.clip(0.4 * heatmap + 0.6 * original_image, 0, 255).astype(np.uint8)

    # Plot images
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    plt.title(f"Original Image\nPredicted: {class_names[predicted_class]}")
    plt.imshow(original_image)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM Overlay")
    plt.imshow(overlay)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Function to process test folder
def process_test_folder(test_folder_path, model):
    target_layer = model.vgg.features[-1]  # Last convolutional layer

    for subfolder in os.listdir(test_folder_path):
        subfolder_path = os.path.join(test_folder_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue

        print(f"\nProcessing folder: {subfolder_path}")
        image_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        image_files = image_files[:MAX_IMAGES_PER_FOLDER]

        for file in image_files:
            image_path = os.path.join(subfolder_path, file)
            print(f"\nProcessing: {image_path}")
            try:
                image_tensor = preprocess_image(image_path)

                # Generate Grad-CAM
                gradcam, predicted_class_idx = generate_gradcam(model, image_tensor, target_layer)

                # Visualize
                visualize_results(image_path, gradcam, predicted_class_idx)
            except Exception as e:
                print(f"Error processing {image_path}: {e}")

# Execute the pipeline
process_test_folder(test_folder_path, model)


Output hidden; open in https://colab.research.google.com to view.